# Three-Model Evaluation

Compare Transformer, LSTM, and FCN checkpoints on the same test groups and plot the shared ground truth once per group.

## Load libraries

In [ ]:
import numpy as np
import re
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import (
    root_mean_squared_error,
    r2_score
)

from shaft_force_sensing.training.utils import load_model
from shaft_force_sensing.evaluation import tb_to_numpy, array_bais, array_medfilt

%load_ext autoreload
%autoreload 2

# Helpers

In [ ]:
METRICS_PATTERN = re.compile(
    r'^(?P<axis>[^:]+):\s*Range=(?P<Range>[-+0-9.eE]+),\s*'
    r'RMSE=(?P<RMSE>[-+0-9.eE]+),\s*'
    r'NRMSE=(?P<NRMSE>[-+0-9.eE]+)%,\s*'
    r'R2=(?P<R2>[-+0-9.eE]+)$'
)

In [ ]:
def parse_metrics_file(path: Path) -> tuple[dict, pd.DataFrame]:
    lines = [ln.strip() for ln in path.read_text(encoding='utf-8').splitlines()]

    if not lines or not lines[0].startswith('Model:'):
        raise ValueError(f'Unexpected metrics format: {path}')

    header = {'Model': lines[0].split(':', 1)[1].strip()}
    header['Test data'] = lines[1].split(':', 1)[1].strip()

    dashed = [i for i, ln in enumerate(lines) if ln == '-' * 10]
    if len(dashed) < 2:
        raise ValueError(f'Cannot locate hparams section in: {path}')

    hparam_lines = lines[dashed[0] + 1 : dashed[1]]
    for ln in hparam_lines:
        if ': ' in ln:
            k, v = ln.split(': ', 1)
            header[k] = v

    rows = []
    current_group = None
    for ln in lines[dashed[1] + 1 :]:
        if not ln or ln == '-' * 10:
            continue
        if ln.startswith('Group: '):
            current_group = ln.split(': ', 1)[1]
            continue

        m = METRICS_PATTERN.match(ln)
        if m and current_group is not None:
            row = {'group': current_group, 'axis': m.group('axis')}
            row.update({
                'Range': float(m.group('Range')),
                'RMSE': float(m.group('RMSE')),
                'NRMSE': float(m.group('NRMSE')),
                'R2': float(m.group('R2')),
            })
            rows.append(row)

    return header, pd.DataFrame(rows)

## Configure checkpoints

In [ ]:
LOG_ROOT = Path("../logs")

GROUP_DIRS = {
    "All inputs": LOG_ROOT / "ablations" / "full",
   r"No HEX10": LOG_ROOT / "ablations" / "no_hex10",
   r"No $\tau$": LOG_ROOT / "ablations" / "no_tau",
   r"No $q$": LOG_ROOT / "ablations" / "no_pos",
   r"No $\dot{q}$": LOG_ROOT / "ablations" / "no_vec",
   r"No $q$ or $\dot{q}$": LOG_ROOT / "ablations" / "no_pos_vec",
}

In [ ]:
eval_dfs = {name: [] for name in GROUP_DIRS.keys()}

for name, dir in GROUP_DIRS.items():
    metrics_files = dir.parent.rglob(f"{dir.name}*/*/metrics.txt")
    for path in tqdm(metrics_files, desc=f"Parsing {name}"):
        _, df = parse_metrics_file(path)
        eval_dfs[name].append(df)
    eval_dfs[name] = pd.concat(eval_dfs[name], ignore_index=True)

In [ ]:
RMSEs = {name: df[df['group'] == 'All'].set_index('axis')['RMSE'] for name, df in eval_dfs.items()}
NRMSEs = {name: df[df['group'] == 'All'].set_index('axis')['NRMSE'] for name, df in eval_dfs.items()}
R2s = {name: df[df['group'] == 'All'].set_index('axis')['R2'] for name, df in eval_dfs.items()}

In [ ]:
axes_to_plot = ['F_x', 'F_y', 'F_z']
ablation_names = list(NRMSEs.keys())
r2_colors = {'F_x': 'red', 'F_y': 'blue', 'F_z': 'green'}

fig = plt.figure(figsize=(5.6 * len(axes_to_plot), 6))
gs = fig.add_gridspec(2, len(axes_to_plot), wspace=0.28, hspace=0.35)

base_positions = np.arange(1, len(ablation_names) + 1)
metric_series = [
    ('NRMSE (%)', NRMSEs),
    ('$R^2$ (%)', R2s),
]
cut_ranges = {
    'NRMSE (%)': (3.0, 4.5),
    '$R^2$ (%)': (75.0, 90.0),
}

row_ref_axes = {0: None, 1: None}


def draw_violin(ax, metric_data, color):
    vp = ax.violinplot(
        metric_data,
        positions=base_positions,
        widths=0.62,
        vert=False,
        showmeans=False,
        showmedians=True,
        showextrema=True,
    )
    for body in vp['bodies']:
        body.set_facecolor(color)
        body.set_edgecolor(color)
        body.set_alpha(0.90)
        body.set_linewidth(1.1)
    if 'cmedians' in vp:
        vp['cmedians'].set_color('black')
        vp['cmedians'].set_linewidth(1.6)
    if 'cbars' in vp:
        vp['cbars'].set_color(color)
        vp['cbars'].set_linewidth(1.0)
    if 'cmins' in vp:
        vp['cmins'].set_color(color)
        vp['cmins'].set_linewidth(1.0)
    if 'cmaxes' in vp:
        vp['cmaxes'].set_color(color)
        vp['cmaxes'].set_linewidth(1.0)


for col_idx, axis_name in enumerate(axes_to_plot):
    color = r2_colors[axis_name]

    for row_idx, (metric_name, metric_map) in enumerate(metric_series):
        share_ax = row_ref_axes[row_idx]

        if axis_name == 'F_z':
            sub_gs = gs[row_idx, col_idx].subgridspec(1, 2, width_ratios=[3, 2], wspace=0.05)
            if share_ax is None:
                ax_left = fig.add_subplot(sub_gs[0, 0])
            else:
                ax_left = fig.add_subplot(sub_gs[0, 0], sharey=share_ax)
            ax_right = fig.add_subplot(sub_gs[0, 1], sharey=ax_left)
            axs_curr = [ax_left, ax_right]
        else:
            if share_ax is None:
                ax = fig.add_subplot(gs[row_idx, col_idx])
                row_ref_axes[row_idx] = ax
            else:
                ax = fig.add_subplot(gs[row_idx, col_idx], sharey=share_ax)
            axs_curr = [ax]

        metric_data = []
        for ablation in ablation_names:
            series = metric_map[ablation]
            if axis_name in series.index:
                vals = series.loc[axis_name]
                if isinstance(vals, pd.Series):
                    vals = vals.dropna().to_numpy(dtype=float)
                else:
                    vals = np.array([vals], dtype=float)
            else:
                vals = np.array([np.nan], dtype=float)

            if vals.size == 0:
                vals = np.array([np.nan], dtype=float)

            metric_data.append(vals)

        metric_all = []
        for vals in metric_data:
            finite_vals = vals[np.isfinite(vals)]
            if finite_vals.size:
                metric_all.append(finite_vals)

        if metric_all:
            metric_all = np.concatenate(metric_all)
            metric_min = float(np.nanmin(metric_all))
            metric_max = float(np.nanmax(metric_all))
        else:
            metric_min, metric_max = 0.0, 1.0

        if metric_name == 'NRMSE (%)':
            metric_min = np.floor(metric_min / 0.25) * 0.25
            metric_max = np.ceil(metric_max / 0.25) * 0.25
            if np.isclose(metric_min, metric_max):
                metric_max = metric_min + 0.25
            tick_formatter = lambda value: f'{value:.2f}'
        else:
            metric_min = float(np.floor(metric_min))
            metric_max = float(np.ceil(metric_max))
            if np.isclose(metric_min, metric_max):
                metric_max = metric_min + 1.0
            tick_formatter = lambda value: f'{int(value)}'

        if axis_name == 'F_z':
            cut_lo, cut_hi = cut_ranges[metric_name]
            left_min, left_max = metric_min, cut_lo
            right_min, right_max = cut_hi, metric_max

            if left_min >= left_max:
                left_min = cut_lo - (0.5 if metric_name == 'NRMSE (%)' else 5.0)
            if right_min >= right_max:
                right_max = cut_hi + (0.5 if metric_name == 'NRMSE (%)' else 5.0)

            for ax_piece in axs_curr:
                draw_violin(ax_piece, metric_data, color)
                ax_piece.set_yticks(base_positions)
                ax_piece.grid(axis='x', alpha=0.3)
                ax_piece.tick_params(axis='x', colors='black')
                ax_piece.tick_params(axis='y', colors='black')
                for spine in ax_piece.spines.values():
                    spine.set_color('black')

            ax_left, ax_right = axs_curr
            ax_left.set_xlim(left_min, left_max)
            ax_right.set_xlim(right_min, right_max)

            # Place inner ticks at offset positions and show offset values.
            inner_shift = 0.25 if metric_name == 'NRMSE (%)' else 1.0
            left_margin = 0.03 * (left_max - left_min)
            right_margin = 0.03 * (right_max - right_min)
            left_inner_x = max(left_min + left_margin, left_max - inner_shift)
            right_inner_x = min(right_max - right_margin, right_min + inner_shift)

            ax_left.set_xticks([left_min, left_inner_x])
            ax_right.set_xticks([right_inner_x, right_max])
            ax_left.set_xticklabels([tick_formatter(left_min), tick_formatter(left_inner_x)])
            ax_right.set_xticklabels([tick_formatter(right_inner_x), tick_formatter(right_max)])

            ax_left.spines['right'].set_visible(False)
            ax_right.spines['left'].set_visible(False)
            ax_right.tick_params(axis='y', left=False, labelleft=False)

            d = 0.012
            kwargs = dict(transform=ax_left.transAxes, color='black', clip_on=False, linewidth=1.0)
            ax_left.plot((1 - d, 1 + d), (-d, +d), **kwargs)
            ax_left.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)
            kwargs.update(transform=ax_right.transAxes)
            ax_right.plot((-d, +d), (-d, +d), **kwargs)
            ax_right.plot((-d, +d), (1 - d, 1 + d), **kwargs)

            if col_idx == 0:
                ax_left.set_yticklabels(ablation_names, color='black')
            else:
                ax_left.tick_params(axis='y', labelleft=False)

            ax_left.set_xlabel(metric_name, color='black')
            ax_right.set_xlabel('')

            if row_idx == 0:
                ax_left.set_title(f'${axis_name}$', color='black')
        else:
            ax = axs_curr[0]
            draw_violin(ax, metric_data, color)

            ax.set_xlim(metric_min, metric_max)
            ax.set_xticks([metric_min, metric_max])
            ax.set_xticklabels([tick_formatter(metric_min), tick_formatter(metric_max)])

            ax.set_yticks(base_positions)
            if col_idx == 0:
                ax.set_yticklabels(ablation_names, color='black')
            else:
                ax.tick_params(axis='y', labelleft=False)

            ax.set_xlabel(metric_name, color='black')
            ax.grid(axis='x', alpha=0.3)
            ax.tick_params(axis='x', colors='black')
            ax.tick_params(axis='y', colors='black')

            for spine in ax.spines.values():
                spine.set_color('black')

            if row_idx == 0:
                ax.set_title(f'${axis_name}$', color='black')

fig.supylabel('Ablation', color='black')
# fig.suptitle('NRMSE and R2 Violin Plots Across Ablations', color='black')
plt.tight_layout(rect=[0.03, 0.0, 1.0, 0.96])

out_dir = Path('../logs/results')
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / 'ablation_nrmse_r2_violin_box.pdf', bbox_inches='tight')

plt.show()